# Chapter 17 — Building the Baseline Email Agent

Hands-On LangChain. Code targets **LangGraph 1.x**.

Before we can give an agent memory, we need an agent. In this chapter we build a baseline **email
assistant** — one that can't yet learn, but establishes the architecture we'll add memory to throughout
Part IV. It has two parts: a **triage router** that classifies each incoming email (ignore / notify /
respond), and a **main response agent** that uses tools to act on the ones that need a reply. We wire
them together with LangGraph.

Companion notebook for Chapter 17. The prompt templates live in `prompts.py` (next to this notebook); the
Ollama chat model comes from `MODEL` in `.env`.

```bash
pip install langgraph langmem langchain langchain-ollama python-dotenv -q
```

## 17.1 Setting the stage: profiles and instructions

First we give the agent context: **who** it works for and the **rules** it follows. We load the local
configuration, then define a user **profile** and a set of **instructions**. We keep the instructions
separate from the prompts on purpose — it makes the system modular, and (importantly for this part) lets
us generate some of them dynamically from memory later.

In [1]:
import os
import warnings
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv(usecwd=True))
warnings.filterwarnings("ignore")   # quiet LangGraph/LangChain pending-deprecation notices

profile = {
    "name": "Youssef",
    "full_name": "Youssef Hosni",
    "user_profile_background": "Senior Data Scientist leading a team of 5 AI engineers.",
}

prompt_instructions = {
    "triage_rules": {
        "ignore": "Marketing newsletters, spam emails, mass company announcements.",
        "notify": "Team member out sick, build system notifications, project status updates.",
        "respond": "Direct questions from team members, meeting requests, critical bug reports.",
    },
    "agent_instructions": "Use these tools when appropriate to help manage Youssef's tasks efficiently.",
}

Here's a sample email to work with — it shows the input schema (`from`, `to`, `subject`, `body`) and gives us something concrete to test.

In [2]:
email = {
    "from": "Oliver Jack <Oliver.Jack@ToDataBeyond.com>",
    "to": "Youssef Hosni <Youssef.Hosni@ToDataBeyond.com>",
    "subject": "Quick question about API documentation",
    "body": """Hi Youssef,

I was reviewing the API documentation for the new authentication service and noticed a few
endpoints seem to be missing from the specs. Could you clarify whether this was intentional?
Specifically, I'm looking at:
- /auth/refresh
- /auth/validate

Thanks!
Oliver""",
}

## 17.2 The triage agent — an intelligent router

The first step in the pipeline is **triage**: we don't want the agent wasting effort on spam or
FYI-only mail. We define a Pydantic `Router` schema that forces the model to return both a
classification *and* its reasoning, then bind it with `with_structured_output`.

In [3]:
from pydantic import BaseModel, Field
from typing_extensions import Literal
from model_provider import get_chat_model

llm = get_chat_model()

class Router(BaseModel):
    """Analyze the unread email and route it according to its content."""
    reasoning: str = Field(description="Step-by-step reasoning behind the classification.")
    classification: Literal["ignore", "respond", "notify"] = Field(
        description="The classification of an email: 'ignore' for irrelevant emails, "
        "'notify' for important information that doesn't need a response, "
        "'respond' for emails that need a reply",
    )

llm_router = llm.with_structured_output(Router)

We instruct the model with prompt templates from `prompts.py`: a system prompt (role, background, rules) and a user prompt (the raw email). Then we invoke the router.

In [ ]:
from prompts import triage_system_prompt, triage_user_prompt

system_prompt = triage_system_prompt.format(
    full_name=profile["full_name"],
    name=profile["name"],
    examples=None,  # we'll add few-shot examples later, using memory
    user_profile_background=profile["user_profile_background"],
    triage_no=prompt_instructions["triage_rules"]["ignore"],
    triage_notify=prompt_instructions["triage_rules"]["notify"],
    triage_email=prompt_instructions["triage_rules"]["respond"],
)
user_prompt = triage_user_prompt.format(
    author=email["from"], to=email["to"], subject=email["subject"], email_thread=email["body"],
)

result = llm_router.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
])
print("classification:", result.classification)
print("reasoning:", result.reasoning)

## 17.3 The main agent — the doer

When triage flags an email as `respond`, it goes to the main agent — the "doer", equipped with tools to
take real actions. For now the three tools are placeholders (in a real app they'd call Gmail / Google
Calendar).

In [5]:
from langchain_core.tools import tool

@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Write and send an email."""
    return f"Email sent to {to} with subject '{subject}'"

@tool
def schedule_meeting(attendees: list[str], subject: str,
                     duration_minutes: int, preferred_day: str) -> str:
    """Schedule a calendar meeting."""
    return f"Meeting '{subject}' scheduled for {preferred_day} with {len(attendees)} attendees"

@tool
def check_calendar_availability(day: str) -> str:
    """Check calendar availability for a given day."""
    return f"Available times on {day}: 9:00 AM, 2:00 PM, 4:00 PM"

We assemble the agent with LangGraph's prebuilt **`create_react_agent`**, giving it a model, the tools, and a prompt function that builds the system prompt from the current state.

In [6]:
import warnings; warnings.filterwarnings("ignore")
from prompts import agent_system_prompt
from langgraph.prebuilt import create_react_agent

# The chat model is shared with the structured triage router.
def create_prompt(state):
    return [
        {"role": "system",
         "content": agent_system_prompt.format(
             instructions=prompt_instructions["agent_instructions"], **profile)}
    ] + state["messages"]

tools = [write_email, schedule_meeting, check_calendar_availability]
agent = create_react_agent(
    get_chat_model(),
    tools=tools,
    prompt=create_prompt,
)

A quick test that the agent can use its tools:

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is my availability for tuesday?"}]}
)
response["messages"][-1].pretty_print()

## 17.4 The grand assembly — the email agent graph

Now we combine the triage router and the response agent into one LangGraph application. The **state**
holds the incoming `email_input` and the running `messages`. The **`triage_router`** node is the entry
point: it classifies the email and uses a `Command` to route either to the `response_agent` or to `END`.

In [8]:
from typing_extensions import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.types import Command
from typing import Literal as TLiteral

class State(TypedDict):
    email_input: dict
    messages: Annotated[list, add_messages]

def triage_router(state: State) -> Command[TLiteral["response_agent", "__end__"]]:
    e = state["email_input"]
    system_prompt = triage_system_prompt.format(
        full_name=profile["full_name"], name=profile["name"], examples=None,
        user_profile_background=profile["user_profile_background"],
        triage_no=prompt_instructions["triage_rules"]["ignore"],
        triage_notify=prompt_instructions["triage_rules"]["notify"],
        triage_email=prompt_instructions["triage_rules"]["respond"],
    )
    user_prompt = triage_user_prompt.format(
        author=e["from"], to=e["to"], subject=e["subject"], email_thread=e["body"],
    )
    result = llm_router.invoke([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ])
    if result.classification == "respond":
        print("📧 Classification: RESPOND - This email requires a response")
        goto = "response_agent"
        update = {"messages": [{"role": "user", "content": f"Respond to the email {e}"}]}
    elif result.classification == "ignore":
        print("🚫 Classification: IGNORE - This email can be safely ignored")
        goto, update = END, None
    else:
        print("🔔 Classification: NOTIFY - This email contains important information")
        goto, update = END, None
    return Command(goto=goto, update=update)

In [9]:
email_agent = StateGraph(State)
email_agent.add_node("triage_router", triage_router)
email_agent.add_node("response_agent", agent)
email_agent.add_edge(START, "triage_router")
email_agent = email_agent.compile()

Let's see the structure of the compiled graph:

In [10]:
print(email_agent.get_graph(xray=True).draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	triage_router(triage_router)
	__end__([<p>__end__</p>]):::last
	__start__ --> triage_router;
	triage_router -.-> __end__;
	triage_router -.-> response_agent\3a__start__;
	response_agent\3a__end__ --> __end__;
	subgraph response_agent
	response_agent\3a__start__(<p>__start__</p>)
	response_agent\3aagent(agent)
	response_agent\3atools(tools)
	response_agent\3a__end__(<p>__end__</p>)
	response_agent\3a__start__ --> response_agent\3aagent;
	response_agent\3aagent -.-> response_agent\3a__end__;
	response_agent\3aagent -.-> response_agent\3atools;
	response_agent\3atools --> response_agent\3aagent;
	end
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 17.5 Testing the full system

**Scenario 1 — a spam email.** A marketing blast that should be ignored.

In [ ]:
spam = {
    "from": "Marketing Team <marketing@amazingdeals.com>",
    "to": "Youssef Hosni <Youssef.Hosni@ToDataBeyond.com>",
    "subject": "EXCLUSIVE OFFER: Limited Time Discount on Developer Tools!",
    "body": "Dear Developer, for the NEXT 24 HOURS enjoy an EXCLUSIVE 75% DISCOUNT on our "
            "Premium Developer Suite! Claim your offer now.",
}
_ = email_agent.invoke({"email_input": spam})

**Scenario 2 — an actionable email** from a colleague, which needs a real response.

In [ ]:
response = email_agent.invoke({"email_input": email})

In [ ]:
for m in response["messages"]:
    m.pretty_print()

The agent triaged the email as `respond`, engaged the main agent, used the `write_email` tool to
draft and "send" a reply, and summarized what it did.

We've built a functional — but **memory-less** — email assistant: a multi-stage LangGraph agent that
classifies mail and acts on it. Its limitation is that it can't learn from past interactions. In the next
chapter we begin integrating **LangMem** to give it **semantic memory**, so it can remember facts about
the user and their contacts.